# Algorithms Homework 4: Constrained Optimality Conditions

## Course policies and AI category

Read the [Artificial Intelligence Policy](../../org/syllabus.md#artificial-intelligence-policy) and
[Collaboration Policy and Honor Code](../../org/syllabus.md#collaboration-policy-and-honor-code)
before starting. Assignment-specific directions control. The categories are **No AI**, **AI
permitted after independent work**, and **AI required**.

Unless a problem says otherwise, its category is **AI permitted after independent work**. Spend about
30 minutes on each top-level problem without AI, solution pages, or another person's help, stopping
early if complete. You may consult lecture notes, textbooks, and nonsolution pages of the course
website; bias toward those course sources. Afterward, AI and genuine collaboration, including coding
together, are permitted. Everyone must contribute intellectually, understand the work, and verify it.

At the end of each top-level problem, add a concise **AI and independent-work report**: approximately
how long the independent attempt took, how far you got, where you became stuck, any AI or collaborative
help used afterward, and how you verified it. If you used no AI, say so. Do not submit prompts or
transcripts. Time estimates help the instructor improve the assignment and are not a speed test.


## Overview

This assignment covers nonlinear constrained optimization, specifically:
* Lecture 18: Introduction to Constrained Optimization
* Lecture 19: KKT Conditions, Multipliers, and Sensitivity
* Lecture 20: Constraint Qualifications

You may do the derivations on paper; upload a photo or scan of that work with your notebook. The
Python cells are short.

**Sign convention.** Use the convention from Lectures 18 and 19 throughout:

$$
\min_{\mathbf{x}} f(\mathbf{x}) \quad \text{s.t.} \quad \mathbf{g}(\mathbf{x}) \le \mathbf{0}, \quad \mathbf{h}(\mathbf{x}) = \mathbf{0},
\qquad
L(\mathbf{x}, \mathbf{u}, \mathbf{v}) = f(\mathbf{x}) + \mathbf{u}^T \mathbf{g}(\mathbf{x}) + \mathbf{v}^T \mathbf{h}(\mathbf{x}),
$$

with $\mathbf{u} \ge \mathbf{0}$. Problem 2 asks how Pyomo's sign convention for duals relates to this one.

In [ ]:
### Load Python libraries

import sys

if "google.colab" in sys.modules:
    !wget "https://raw.githubusercontent.com/ndcbe/optimization/main/notebooks/helper.py"
    import helper

    helper.easy_install()
else:
    sys.path.insert(0, "../")
    import helper
helper.set_plotting_style()

import numpy as np
import sympy as sym
import pyomo.environ as pyo

x1, x2 = sym.symbols("x1 x2", real=True)

## Problem 1: Convexity, local minimizers and second-order conditions

Recall that a nonlinear program is convex when:
* the objective is convex
* every inequality $g_i(\mathbf{x}) \le 0$ has a convex $g_i$, and
* every equality is affine.

Then every local minimizer is a global minimizer. The test is **sufficient,
not necessary**: a problem can fail it and still be convex.

### 1a. Is it convex?

For each problem below, decide whether the test certifies convexity. If it does not, decide whether
the problem is nonetheless convex. To show a feasible set is **not** convex, give two feasible points
whose midpoint is infeasible.

**1a(i).** $\min (x_1 - 2)^2 + (x_2 - 2)^2$ s.t. $x_1^2 + x_2^2 \le 4$, $x_1 + x_2 = 1$.

**Answer:**

**1a(ii).** $\min x_1 + x_2$ s.t. $x_1^2 + x_2^2 = 1$.

**Answer:**

**1a(iii).** $\min x_1 + x_2$ s.t. $1 - x_1 x_2 \le 0$, $x_1 > 0$, $x_2 > 0$. *Hint:* is
$1 - x_1 x_2$ convex? Can the same feasible set be written with a convex constraint function?

In [ ]:
# Add your solution here

**Answer:**

### 1b. Classify the KKT points of the circle problem

For $\min x_1 + x_2$ s.t. $h(\mathbf{x}) = x_1^2 + x_2^2 - 1 = 0$:

1. Find every KKT point and its multiplier $v$, with $L = f + v\,h$.
2. At each one, find the critical directions (here, directions $\mathbf{d}$ with
   $\nabla h(\mathbf{x}^*)^T \mathbf{d} = 0$) and evaluate $\mathbf{d}^T \nabla^2_{\mathbf{xx}} L \, \mathbf{d}$ on them.
3. Use the second-order conditions to classify each point. Which is the global
   minimizer, and how do you know a global minimizer exists?

In [ ]:
# Add your solution here

**Answer:**

### 1c. Replace the equality by an inequality

Now solve $\min x_1 + x_2$ s.t. $x_1^2 + x_2^2 - 1 \le 0$. Is this problem convex? Which of the two
points from 1b is still a KKT point, and which condition does the other one fail?

**Answer:**

### 1d. What does a local solver return?

The code below solves the equality-constrained circle problem with Ipopt from two starting points.
Run it. Which point does each run return, and what termination condition does Ipopt report? Explain
the run from $(1, 1)$ using 1b. *Hint:* what happens to $x_1 - x_2$ along the iterates?

In [ ]:
for start in [(1.0, 1.0), (0.5, 0.9)]:
    m = pyo.ConcreteModel()
    m.x1 = pyo.Var(initialize=start[0])
    m.x2 = pyo.Var(initialize=start[1])
    m.obj = pyo.Objective(expr=m.x1 + m.x2)
    m.h = pyo.Constraint(expr=m.x1**2 + m.x2**2 == 1)
    res = pyo.SolverFactory("ipopt").solve(m)
    print(f"start {start}: {res.solver.termination_condition},  "
          f"x = ({pyo.value(m.x1):.4f}, {pyo.value(m.x2):.4f}),  f = {pyo.value(m.obj):.4f}")

**Answer:**

### AI and independent-work report

Approximately how long was your independent attempt, how far did you get, where did you get stuck,
what AI or collaborative help did you use afterward, and how did you verify it? If you used no AI, say so.

## Problem 2: KKT conditions for a small nonlinear program

Consider

$$
\begin{aligned}
\min_{\mathbf{x}} \quad & f(\mathbf{x}) = (x_1 - 2)^2 + (x_2 - 1)^2 \\
\mathrm{s.t.} \quad & g_1(\mathbf{x}) = x_1^2 - x_2 \le 0, \\
& g_2(\mathbf{x}) = x_1 + x_2 - 2 \le 0.
\end{aligned}
$$

Sketch the feasible region and a few contours of $f$ before you start. It will help with every part.

### 2a. Write the KKT conditions

Write the Lagrangian and all four groups of KKT conditions for this problem: stationarity, primal
feasibility, dual feasibility and complementarity. Write them component by component, not only in
vector form.

In [ ]:
# Add your solution here

**Answer:**

### 2b. Check the candidate $\mathbf{x} = (1, 1)^T$

Which constraints are active at $\mathbf{x} = (1, 1)^T$? Solve the stationarity equations for the
multipliers. Is $(1, 1)^T$ a KKT point? Justify each of the four groups of conditions.

In [ ]:
# Add your solution here

**Answer:**

### 2c. Check the candidate $\mathbf{x} = (0, 0)^T$

Is $\mathbf{x} = (0, 0)^T$ feasible? Which constraints are active? Is it a KKT point? If not, which
condition fails, and what does $-\nabla f(\mathbf{0})$ tell you about a better direction to move?

In [ ]:
# Add your solution here

**Answer:**

### 2d. Check with Pyomo and Ipopt

Complete the model below, solve it with Ipopt, and compare the solution to 2b. Pyomo stores the
constraint duals in the `dual` suffix. Compare the reported duals to your $u_1, u_2$: what is the
relationship between Pyomo's sign convention and the one in Lecture 19?

In [ ]:
def build_problem2(epsilon=0.0):
    '''Problem 2, with the second constraint perturbed to g2(x) + epsilon <= 0 (used in Problem 3).'''
    m = pyo.ConcreteModel()
    m.x1 = pyo.Var(initialize=0.0)
    m.x2 = pyo.Var(initialize=0.0)
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

    # Add your solution here
    return m


m1 = build_problem2()
results = pyo.SolverFactory("ipopt").solve(m1)
print("termination condition:", results.solver.termination_condition)
print("x* =", pyo.value(m1.x1), pyo.value(m1.x2), "  f* =", pyo.value(m1.obj))
print("Pyomo duals: g1 ->", m1.dual[m1.g1], "  g2 ->", m1.dual[m1.g2])

**Answer:**

### AI and independent-work report

Approximately how long was your independent attempt, how far did you get, where did you get stuck,
what AI or collaborative help did you use afterward, and how did you verify it? If you used no AI, say so.

## Problem 3: Multipliers as sensitivities

Perturb the second constraint of Problem 2 as in Lecture 19:

$$
g_2(\mathbf{x}) + \epsilon = x_1 + x_2 - 2 + \epsilon \le 0.
$$

Positive $\epsilon$ tightens the constraint; negative $\epsilon$ relaxes it.

### 3a. Predict the optimal objective

Using only your multipliers from Problem 2, predict the optimal objective $f^*(\epsilon)$ for
$\epsilon = -0.1$, $0.1$ and $0.5$. State the first-order formula you used.

**Answer:**

### 3b. Re-solve and compare

Re-solve with `build_problem2(epsilon)` for the three values of $\epsilon$. Tabulate the predicted
value, the re-solved value and the difference.

In [ ]:
# Add your solution here

**Answer:**

### 3c. Interpret

1. Why does the error in the prediction grow as $|\epsilon|$ grows?
2. Is the linear prediction above or below the re-solved value for both signs of $\epsilon$? What
   does that tell you about the shape of $f^*(\epsilon)$ near $\epsilon = 0$?

In [ ]:
# Add your solution here

**Answer:**

### AI and independent-work report

Approximately how long was your independent attempt, how far did you get, where did you get stuck,
what AI or collaborative help did you use afterward, and how did you verify it? If you used no AI, say so.

## Problem 4: Constraint qualifications at three points

Consider three problems, each with a known minimizer $\mathbf{x}^*$.

**Case A.** Problem 2 at $\mathbf{x}^* = (1, 1)^T$.

**Case B.** A disk with a tangent line, at $\mathbf{x}^* = (1, 2)^T$:

$$
\min_{\mathbf{x}} \; -x_1 - 2x_2 \quad \mathrm{s.t.} \quad g_1 = x_1^2 + x_2^2 - 5 \le 0, \quad g_2 = x_1 + 2x_2 - 5 \le 0.
$$

**Case C.** Two disks that touch at one point, at $\mathbf{x}^* = (0, 0)^T$:

$$
\min_{\mathbf{x}} \; x_1 + x_2 \quad \mathrm{s.t.} \quad g_1 = (x_1 - 1)^2 + x_2^2 - 1 \le 0, \quad g_2 = (x_1 + 1)^2 + x_2^2 - 1 \le 0.
$$

Sketch Cases B and C before you start.

### 4a. LICQ

For each case, list the active constraints, write the matrix of active constraint gradients at
$\mathbf{x}^*$, and decide whether LICQ holds (Lecture 20).

In [ ]:
# Add your solution here

**Answer:**

### 4b. MFCQ

For each case, either give a direction $\mathbf{d}$ with $\nabla g_i(\mathbf{x}^*)^T \mathbf{d} < 0$
for every active inequality, or explain why no such direction exists.

**Answer:**

### 4c. KKT multipliers

For each case, find all KKT multipliers at $\mathbf{x}^*$. Are they unique, a set (describe it), or
nonexistent? Connect your answer to the theorems in Lecture 20.

**Answer:**

### 4d. What does a solver see?

The code below solves Case B from three starting points and Case C from one, reporting the
termination condition, the solution, and the multipliers in the Lecture 19 sign convention. Run it.
Solver versions differ, so report the values **your** run produces.

In [ ]:
def solve_case(case, x_init):
    m = pyo.ConcreteModel()
    m.x1 = pyo.Var(initialize=x_init[0])
    m.x2 = pyo.Var(initialize=x_init[1])
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    if case == "B":
        m.obj = pyo.Objective(expr=-m.x1 - 2 * m.x2)
        m.g1 = pyo.Constraint(expr=m.x1**2 + m.x2**2 - 5 <= 0)
        m.g2 = pyo.Constraint(expr=m.x1 + 2 * m.x2 - 5 <= 0)
    else:
        m.obj = pyo.Objective(expr=m.x1 + m.x2)
        m.g1 = pyo.Constraint(expr=(m.x1 - 1) ** 2 + m.x2**2 - 1 <= 0)
        m.g2 = pyo.Constraint(expr=(m.x1 + 1) ** 2 + m.x2**2 - 1 <= 0)
    results = pyo.SolverFactory("ipopt").solve(m)
    u = (-m.dual[m.g1], -m.dual[m.g2])  # Lecture 19 sign convention
    print(f"Case {case}, start {x_init}: {results.solver.termination_condition}")
    print(f"   x = ({pyo.value(m.x1):.6g}, {pyo.value(m.x2):.6g}),  f = {pyo.value(m.obj):.6g}")
    print(f"   u = ({u[0]:.6g}, {u[1]:.6g})")
    if case == "B":
        print(f"   2*u1 + u2 = {2 * u[0] + u[1]:.6g}")
    print(f"   g = ({pyo.value(m.g1.body):.3g}, {pyo.value(m.g2.body):.3g})")


for start in [(0, 0), (-1, 0), (3, 3)]:
    solve_case("B", start)
solve_case("C", (0.5, 0.5))

### 4e. Interpret the solver output

1. **Case B.** Do the multipliers change with the starting point? Do they satisfy your answer to 4c?
   Why can a solver return different multipliers for the same $\mathbf{x}^*$?
2. **Case C.** What termination condition does Ipopt report? Which numbers in the output tell you
   that something is wrong anyway? Explain them using 4a--4c.
3. Lecture 20 describes degenerate constraints. Which of Cases B and C is degenerate, and is the
   degeneracy local or global?

**Answer:**

### AI and independent-work report

Approximately how long was your independent attempt, how far did you get, where did you get stuck,
what AI or collaborative help did you use afterward, and how did you verify it? If you used no AI, say so.